# 🔄 Transformer for RUL Prediction - Comprehensive Tutorial

## Learning Objectives

By the end of this tutorial, you will:
- Understand how Transformers work for time series RUL prediction
- Learn Transformer architecture and attention mechanism
- Build multiple Transformer model designs for RUL prediction
- Compare different Transformer configurations
- Visualize attention mechanisms for engine degradation
- Apply Transformers to NASA turbofan engine RUL prediction
- Compare Transformer performance with LSTM/RNN

---

## What is Transformer for RUL Prediction?

**Transformers** use **self-attention** to capture temporal patterns in engine sensor sequences, making them powerful for RUL prediction.

### Why Transformer for RUL?

- **Long-range Dependencies**: Can capture degradation patterns across many cycles
- **Parallel Processing**: Faster training than RNNs/LSTMs
- **Attention Mechanism**: Focuses on important sensor readings and timesteps
- **Flexibility**: Can model complex temporal relationships
- **State-of-the-art**: Often outperforms traditional RNNs

### Key Innovation: Self-Attention for Time Series

- **Attention**: Model learns which timesteps and sensors are important
- **Temporal Patterns**: Captures relationships across entire sequence
- **Multi-head Attention**: Multiple attention mechanisms for different patterns
- **Positional Encoding**: Adds temporal position information

### Our Goal:

Build **multiple Transformer architectures** to predict RUL using NASA turbofan engine data and compare their performance!

---

## Step 1: Import Required Libraries

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch, FancyBboxPatch, Arrow
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras for Transformers
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Embedding, Dropout, GlobalAveragePooling1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# For transformer components
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Add

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Visualization style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('ggplot')

sns.set_palette("husl")
%matplotlib inline

print("✅ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

## Step 2: Load NASA Turbofan Engine Data

In [ ]:
# Define data path
data_path = Path('dataset/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData')

# Column names with actual field names
op_settings = ['Altitude', 'Mach', 'TRA']
sensors = [
    'T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
    'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32'
]
column_names = ['unit', 'time'] + op_settings + sensors

def load_data(dataset='FD001'):
    """Load training and test data from CSV files"""
    train_file = data_path / f'train_{dataset}.csv'
    test_file = data_path / f'test_{dataset}.csv'
    rul_file = data_path / f'RUL_{dataset}.csv'
    
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    rul_df = pd.read_csv(rul_file)
    
    return train_df, test_df, rul_df

def calculate_rul_train(df):
    """Calculate RUL (Remaining Useful Life) for training data"""
    df = df.copy()
    df['RUL'] = df.groupby('unit')['time'].transform(lambda x: x.max() - x)
    return df

# Load data for FD001 dataset
train_df, test_df, rul_df = load_data('FD001')
train_df = calculate_rul_train(train_df)

print("✅ NASA Turbofan Engine Data loaded successfully!")
print(f"\\n📊 Training data shape: {train_df.shape}")
print(f"📊 Test data shape: {test_df.shape}")
print(f"🚁 Number of engines in training: {train_df['unit'].nunique()}")
print(f"🚁 Number of engines in test: {test_df['unit'].nunique()}")
print(f"⏱️  RUL range in training: {train_df['RUL'].min()} to {train_df['RUL'].max()} cycles")

## Step 3: Visualize Transformer Architecture for RUL Prediction

Let's understand how Transformers process engine sensor sequences!

In [ ]:
# Visualize Transformer for RUL Prediction
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Transformer Architecture for RUL Prediction', fontsize=16, fontweight='bold')

# Left: How Transformer processes sequences
ax1 = axes[0]
ax1.set_xlim(-0.5, 8)
ax1.set_ylim(-0.5, 6)
ax1.axis('off')
ax1.set_title('Transformer Processing Engine Sensor Sequences', 
              fontsize=13, fontweight='bold', pad=15)

# Time steps
time_labels = ['t-2', 't-1', 't']
colors = ['#FFA500', '#50C878', '#4A90E2']

for t_idx, (t_label, color) in enumerate(zip(time_labels, colors)):
    x_offset = t_idx * 2.5
    
    # Sensor inputs at this timestep
    for i in range(2):
        circle = Circle((x_offset, 0.5 + i*0.5), 0.12, color='#4A90E2', ec='black', lw=1.2)
        ax1.add_patch(circle)
        if i == 0:
            ax1.text(x_offset, 0.5 + i*0.5, f'S{t_label}', ha='center', va='center', 
                    fontsize=8, fontweight='bold')
    
    # Transformer encoder block
    rect = Rectangle((x_offset - 0.4, 2), 0.8, 1.2, fill=False, edgecolor='black', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(x_offset, 2.6, 'Transformer\\nBlock', ha='center', va='center', fontsize=8, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    
    # Attention connections (all-to-all)
    if t_idx < len(time_labels) - 1:
        for other_idx in range(t_idx + 1, len(time_labels)):
            other_x = other_idx * 2.5
            ax1.plot([x_offset + 0.4, other_x - 0.4], [2.6, 2.6], 
                    color='red', alpha=0.4, linewidth=1.5, linestyle='--')
    
    # Output representation
    circle_out = Circle((x_offset + 1.2, 2.6), 0.15, color=color, ec='black', lw=1.5)
    ax1.add_patch(circle_out)
    ax1.text(x_offset + 1.2, 2.6, f'H{t_label}', ha='center', va='center', fontsize=8, fontweight='bold')

# RUL output
circle_rul = Circle((7.5, 2.6), 0.2, color='#FF6B6B', ec='black', lw=2)
ax1.add_patch(circle_rul)
ax1.text(7.5, 2.6, 'RUL', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Connection to RUL
ax1.arrow(6.7, 2.6, 0.6, 0, head_width=0.12, head_length=0.15, 
         fc='green', ec='green', linewidth=2.5, alpha=0.8)

ax1.text(3.75, -0.3, 'Time Steps →', ha='center', fontsize=11, fontweight='bold', style='italic')
ax1.text(0, 1.1, 'Sensor\\nSequences', ha='center', fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

# Right: Attention mechanism
ax2 = axes[1]
ax2.set_xlim(-0.5, 6)
ax2.set_ylim(-0.5, 5)
ax2.axis('off')
ax2.set_title('Self-Attention: Which Timesteps Matter?', 
              fontsize=13, fontweight='bold', pad=15)

# Example: Attention between timesteps
timesteps = ['Cycle 1', 'Cycle 15', 'Cycle 30']
n_ts = len(timesteps)

for i, ts in enumerate(timesteps):
    x_pos = i * 2 + 1
    circle = Circle((x_pos, 1), 0.3, color='#3498DB', ec='black', lw=2)
    ax2.add_patch(circle)
    ax2.text(x_pos, 1, ts, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    
    # Attention connections
    for j in range(n_ts):
        if i != j:
            other_x = j * 2 + 1
            weight = 0.3 if abs(i-j) == 1 else 0.2
            ax2.plot([x_pos, other_x], [1.3, 1.3], 
                    color='red', alpha=weight, linewidth=weight*5)

# Attention explanation
ax2.text(3, 2.5, 'Each timestep attends to\\nall other timesteps', ha='center', fontsize=10, 
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', alpha=0.8))
ax2.text(3, 3.5, 'Stronger connections =\\nMore important relationships', ha='center', fontsize=9, 
        style='italic', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.show()

print("✅ Transformer architecture visualization for RUL created!")
print("\\n💡 Key Concepts:")
print("   - Each timestep (cycle) can attend to all other timesteps")
print("   - Attention learns which cycles are important for RUL")
print("   - Parallel processing (faster than RNNs)")
print("   - Captures long-range degradation patterns")

In [ ]:
# Define Transformer Encoder Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)  # Self-attention
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)  # Residual connection
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)  # Residual connection

print("✅ Transformer components defined!")
print("\\n💡 TransformerBlock Components:")
print("   - Multi-Head Attention: Captures relationships between timesteps")
print("   - Feed-Forward Network: Processes attended information")
print("   - Layer Normalization: Stabilizes training")
print("   - Residual Connections: Helps gradient flow")
print("   - Dropout: Prevents overfitting")

## Step 5: Create Sequences from Engine Data

Transformers need sequences of sensor readings over time!

In [ ]:
def create_sequences(data, sequence_length=30):
    """
    Create sequences for Transformer training from engine data
    
    Parameters:
    -----------
    data : DataFrame
        Training data with sensor readings and RUL
    sequence_length : int
        Number of timesteps (cycles) to use for prediction
    
    Returns:
    --------
    X_seq : array, shape (n_sequences, sequence_length, n_features)
        Input sequences (sensor readings over time)
    y_seq : array, shape (n_sequences,)
        Target RUL values
    """
    sequences = []
    targets = []
    
    # Select features (sensors + operational settings)
    feature_cols = op_settings + sensors
    
    # For each engine, create sequences
    for unit_id in data['unit'].unique():
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        unit_rul = unit_data['RUL'].values
        
        # Create sequences using sliding window
        for i in range(len(unit_data) - sequence_length + 1):
            sequences.append(unit_features[i:i+sequence_length])
            targets.append(unit_rul[i+sequence_length-1])  # RUL at end of sequence
    
    return np.array(sequences), np.array(targets)

# Create sequences
sequence_length = 30  # Use 30 cycles to predict RUL
X_train_seq, y_train_seq = create_sequences(train_df, sequence_length)

print("✅ Sequences created from NASA turbofan engine data!")
print(f"\\n📊 Sequence Statistics:")
print(f"   - Number of sequences: {X_train_seq.shape[0]:,}")
print(f"   - Sequence length: {X_train_seq.shape[1]} cycles")
print(f"   - Features per timestep: {X_train_seq.shape[2]}")
print(f"   - Data shape: (sequences, timesteps, features) = {X_train_seq.shape}")
print(f"   - Target shape: {y_train_seq.shape}")
print(f"   - RUL range: [{y_train_seq.min()}, {y_train_seq.max()}] cycles")

## Step 6: Scale Features and Split Data

In [ ]:
# Scale the features (crucial for Transformers!)
n_samples, n_timesteps, n_features = X_train_seq.shape
X_reshaped = X_train_seq.reshape(-1, n_features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped)

# Reshape back to sequences
X_train_seq_scaled = X_scaled.reshape(n_samples, n_timesteps, n_features)

# Split into training and validation sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_seq_scaled, y_train_seq, test_size=0.2, random_state=42
)

print("✅ Features scaled and data split!")
print(f"\\n📊 Final Data Shapes:")
print(f"   - Training sequences: {X_train_split.shape[0]:,}")
print(f"   - Validation sequences: {X_val_split.shape[0]:,}")
print(f"   - Sequence length: {n_timesteps} cycles")
print(f"   - Features per timestep: {n_features}")

## Step 7: Model Design 1 - Single Transformer Block

Let's start with a simple single-block Transformer!

In [ ]:
# Model Design 1: Single Transformer Block
embed_dim_1 = 64
num_heads_1 = 4
ff_dim_1 = 128

inputs_1 = layers.Input(shape=(n_timesteps, n_features))

# Dense projection to embedding dimension
x = Dense(embed_dim_1)(inputs_1)

# Positional encoding (learned)
positions = tf.range(start=0, limit=n_timesteps, delta=1)
position_embedding = Embedding(input_dim=n_timesteps, output_dim=embed_dim_1)(positions)
x = x + position_embedding

# Single Transformer block
x = TransformerBlock(embed_dim_1, num_heads_1, ff_dim_1)(x)

# Global pooling and output
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.1)(x)
outputs_1 = Dense(1)(x)  # RUL output

model_1_transformer = Model(inputs=inputs_1, outputs=outputs_1)

model_1_transformer.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

print("📊 Model Design 1: Single Transformer Block")
print("=" * 60)
model_1_transformer.summary()

print("\\n💡 Architecture:")
print("   - 1 Transformer block (4 attention heads)")
print("   - Embedding dimension: 64")
print("   - Simple and fast")
print("   - Good baseline")

In [ ]:
# Train Model 1
print("🚀 Training Model 1: Single Transformer Block...")
print("=" * 60)

history_1_transformer = model_1_transformer.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 1
y_pred_1_transformer = model_1_transformer.predict(X_val_split, verbose=0)

mse_1_transformer = mean_squared_error(y_val_split, y_pred_1_transformer)
mae_1_transformer = mean_absolute_error(y_val_split, y_pred_1_transformer)
rmse_1_transformer = np.sqrt(mse_1_transformer)
r2_1_transformer = r2_score(y_val_split, y_pred_1_transformer)

print("📊 Model 1 Performance (Single Transformer Block):")
print("=" * 60)
print(f"MSE:  {mse_1_transformer:.2f}")
print(f"MAE:  {mae_1_transformer:.2f} cycles")
print(f"RMSE: {rmse_1_transformer:.2f} cycles")
print(f"R²:   {r2_1_transformer:.4f}")

## Step 8: Model Design 2 - Two Transformer Blocks (Stacked)

Stacked Transformers can learn more complex patterns!

In [ ]:
# Model Design 2: Two Transformer Blocks (Stacked)
embed_dim_2 = 64
num_heads_2 = 4
ff_dim_2 = 128

inputs_2 = layers.Input(shape=(n_timesteps, n_features))

# Dense projection
x = Dense(embed_dim_2)(inputs_2)

# Positional encoding
positions = tf.range(start=0, limit=n_timesteps, delta=1)
position_embedding = Embedding(input_dim=n_timesteps, output_dim=embed_dim_2)(positions)
x = x + position_embedding

# Two Transformer blocks
x = TransformerBlock(embed_dim_2, num_heads_2, ff_dim_2)(x)
x = TransformerBlock(embed_dim_2, num_heads_2, ff_dim_2)(x)

# Global pooling and output
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.1)(x)
outputs_2 = Dense(1)(x)

model_2_transformer = Model(inputs=inputs_2, outputs=outputs_2)

model_2_transformer.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

print("📊 Model Design 2: Two Transformer Blocks (Stacked)")
print("=" * 60)
model_2_transformer.summary()

print("\\n💡 Architecture:")
print("   - 2 Transformer blocks (stacked)")
print("   - Each block has 4 attention heads")
print("   - More capacity to learn complex patterns")
print("   - Better for complex degradation patterns")

In [ ]:
# Train Model 2
print("🚀 Training Model 2: Two Transformer Blocks...")
print("=" * 60)

history_2_transformer = model_2_transformer.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 2
y_pred_2_transformer = model_2_transformer.predict(X_val_split, verbose=0)

mse_2_transformer = mean_squared_error(y_val_split, y_pred_2_transformer)
mae_2_transformer = mean_absolute_error(y_val_split, y_pred_2_transformer)
rmse_2_transformer = np.sqrt(mse_2_transformer)
r2_2_transformer = r2_score(y_val_split, y_pred_2_transformer)

print("📊 Model 2 Performance (Two Transformer Blocks):")
print("=" * 60)
print(f"MSE:  {mse_2_transformer:.2f}")
print(f"MAE:  {mae_2_transformer:.2f} cycles")
print(f"RMSE: {rmse_2_transformer:.2f} cycles")
print(f"R²:   {r2_2_transformer:.4f}")

## Step 9: Model Design 3 - Three Transformer Blocks (Deep)

Deep Transformers for maximum pattern recognition!

In [ ]:
# Model Design 3: Three Transformer Blocks (Deep)
embed_dim_3 = 64
num_heads_3 = 4
ff_dim_3 = 128

inputs_3 = layers.Input(shape=(n_timesteps, n_features))

# Dense projection
x = Dense(embed_dim_3)(inputs_3)

# Positional encoding
positions = tf.range(start=0, limit=n_timesteps, delta=1)
position_embedding = Embedding(input_dim=n_timesteps, output_dim=embed_dim_3)(positions)
x = x + position_embedding

# Three Transformer blocks
x = TransformerBlock(embed_dim_3, num_heads_3, ff_dim_3)(x)
x = TransformerBlock(embed_dim_3, num_heads_3, ff_dim_3)(x)
x = TransformerBlock(embed_dim_3, num_heads_3, ff_dim_3)(x)

# Global pooling and output
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.1)(x)
outputs_3 = Dense(1)(x)

model_3_transformer = Model(inputs=inputs_3, outputs=outputs_3)

model_3_transformer.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

print("📊 Model Design 3: Three Transformer Blocks (Deep)")
print("=" * 60)
model_3_transformer.summary()

print("\\n💡 Architecture:")
print("   - 3 Transformer blocks (deep)")
print("   - Maximum capacity for complex patterns")
print("   - Can capture very complex degradation relationships")
print("   - Most complex architecture")

In [ ]:
# Train Model 3
print("🚀 Training Model 3: Three Transformer Blocks...")
print("=" * 60)

history_3_transformer = model_3_transformer.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 3
y_pred_3_transformer = model_3_transformer.predict(X_val_split, verbose=0)

mse_3_transformer = mean_squared_error(y_val_split, y_pred_3_transformer)
mae_3_transformer = mean_absolute_error(y_val_split, y_pred_3_transformer)
rmse_3_transformer = np.sqrt(mse_3_transformer)
r2_3_transformer = r2_score(y_val_split, y_pred_3_transformer)

print("📊 Model 3 Performance (Three Transformer Blocks):")
print("=" * 60)
print(f"MSE:  {mse_3_transformer:.2f}")
print(f"MAE:  {mae_3_transformer:.2f} cycles")
print(f"RMSE: {rmse_3_transformer:.2f} cycles")
print(f"R²:   {r2_3_transformer:.4f}")

## Step 10: Model Design 4 - Transformer with More Attention Heads

More attention heads can capture different types of patterns!

In [ ]:
# Model Design 4: Transformer with More Attention Heads
embed_dim_4 = 64
num_heads_4 = 8  # More attention heads
ff_dim_4 = 128

inputs_4 = layers.Input(shape=(n_timesteps, n_features))

# Dense projection
x = Dense(embed_dim_4)(inputs_4)

# Positional encoding
positions = tf.range(start=0, limit=n_timesteps, delta=1)
position_embedding = Embedding(input_dim=n_timesteps, output_dim=embed_dim_4)(positions)
x = x + position_embedding

# Two Transformer blocks with more heads
x = TransformerBlock(embed_dim_4, num_heads_4, ff_dim_4)(x)
x = TransformerBlock(embed_dim_4, num_heads_4, ff_dim_4)(x)

# Global pooling and output
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.1)(x)
outputs_4 = Dense(1)(x)

model_4_transformer = Model(inputs=inputs_4, outputs=outputs_4)

model_4_transformer.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

print("📊 Model Design 4: Transformer with 8 Attention Heads")
print("=" * 60)
model_4_transformer.summary()

print("\\n💡 Architecture:")
print("   - 2 Transformer blocks")
print("   - 8 attention heads (vs 4 in previous models)")
print("   - Each head can focus on different patterns")
print("   - More diverse attention patterns")

In [ ]:
# Train Model 4
print("🚀 Training Model 4: Transformer with 8 Attention Heads...")
print("=" * 60)

history_4_transformer = model_4_transformer.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 4
y_pred_4_transformer = model_4_transformer.predict(X_val_split, verbose=0)

mse_4_transformer = mean_squared_error(y_val_split, y_pred_4_transformer)
mae_4_transformer = mean_absolute_error(y_val_split, y_pred_4_transformer)
rmse_4_transformer = np.sqrt(mse_4_transformer)
r2_4_transformer = r2_score(y_val_split, y_pred_4_transformer)

print("📊 Model 4 Performance (8 Attention Heads):")
print("=" * 60)
print(f"MSE:  {mse_4_transformer:.2f}")
print(f"MAE:  {mae_4_transformer:.2f} cycles")
print(f"RMSE: {rmse_4_transformer:.2f} cycles")
print(f"R²:   {r2_4_transformer:.4f}")

## Step 11: Model Design 5 - Transformer with Larger Embedding Dimension

Larger embedding dimensions can capture more information!

In [ ]:
# Model Design 5: Transformer with Larger Embedding Dimension
embed_dim_5 = 128  # Larger embedding dimension
num_heads_5 = 4
ff_dim_5 = 256  # Larger feed-forward dimension

inputs_5 = layers.Input(shape=(n_timesteps, n_features))

# Dense projection
x = Dense(embed_dim_5)(inputs_5)

# Positional encoding
positions = tf.range(start=0, limit=n_timesteps, delta=1)
position_embedding = Embedding(input_dim=n_timesteps, output_dim=embed_dim_5)(positions)
x = x + position_embedding

# Two Transformer blocks
x = TransformerBlock(embed_dim_5, num_heads_5, ff_dim_5)(x)
x = TransformerBlock(embed_dim_5, num_heads_5, ff_dim_5)(x)

# Global pooling and output
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.1)(x)
outputs_5 = Dense(1)(x)

model_5_transformer = Model(inputs=inputs_5, outputs=outputs_5)

model_5_transformer.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

print("📊 Model Design 5: Transformer with Larger Embedding (128)")
print("=" * 60)
model_5_transformer.summary()

print("\\n💡 Architecture:")
print("   - 2 Transformer blocks")
print("   - Embedding dimension: 128 (vs 64)")
print("   - Feed-forward dimension: 256 (vs 128)")
print("   - More capacity to store information")

In [ ]:
# Train Model 5
print("🚀 Training Model 5: Transformer with Larger Embedding...")
print("=" * 60)

history_5_transformer = model_5_transformer.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Model 5
y_pred_5_transformer = model_5_transformer.predict(X_val_split, verbose=0)

mse_5_transformer = mean_squared_error(y_val_split, y_pred_5_transformer)
mae_5_transformer = mean_absolute_error(y_val_split, y_pred_5_transformer)
rmse_5_transformer = np.sqrt(mse_5_transformer)
r2_5_transformer = r2_score(y_val_split, y_pred_5_transformer)

print("📊 Model 5 Performance (Larger Embedding):")
print("=" * 60)
print(f"MSE:  {mse_5_transformer:.2f}")
print(f"MAE:  {mae_5_transformer:.2f} cycles")
print(f"RMSE: {rmse_5_transformer:.2f} cycles")
print(f"R²:   {r2_5_transformer:.4f}")

## Step 12: Compare All Transformer Model Designs

Let's compare all 5 Transformer architectures!

In [ ]:
# Create comprehensive comparison
comparison_data = {
    'Model': [
        '1-Block Transformer',
        '2-Block Transformer',
        '3-Block Transformer',
        '8-Heads Transformer',
        'Large Embedding Transformer'
    ],
    'MSE': [mse_1_transformer, mse_2_transformer, mse_3_transformer, mse_4_transformer, mse_5_transformer],
    'MAE (cycles)': [mae_1_transformer, mae_2_transformer, mae_3_transformer, mae_4_transformer, mae_5_transformer],
    'RMSE (cycles)': [rmse_1_transformer, rmse_2_transformer, rmse_3_transformer, rmse_4_transformer, rmse_5_transformer],
    'R²': [r2_1_transformer, r2_2_transformer, r2_3_transformer, r2_4_transformer, r2_5_transformer]
}

comparison_df = pd.DataFrame(comparison_data)

print("📊 Comprehensive Transformer Model Comparison (Validation Set):")
print("=" * 100)
print(comparison_df.to_string(index=False))

# Find best model
best_model_idx = comparison_df['R²'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
print(f"\\n🏆 Best Model: {best_model_name}")
print(f"   - R²: {comparison_df.loc[best_model_idx, 'R²']:.4f}")
print(f"   - RMSE: {comparison_df.loc[best_model_idx, 'RMSE (cycles)']:.2f} cycles")
print(f"   - MAE: {comparison_df.loc[best_model_idx, 'MAE (cycles)']:.2f} cycles")

## Step 13: Visualize Model Comparison

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Transformer Model Designs Comparison for RUL Prediction', fontsize=16, fontweight='bold')

models = comparison_df['Model'].values
colors = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12', '#9B59B6']

# R² comparison
axes[0, 0].bar(range(len(models)), comparison_df['R²'], color=colors, alpha=0.7, 
               edgecolor='black', linewidth=1.5)
axes[0, 0].set_xticks(range(len(models)))
axes[0, 0].set_xticklabels([m.replace(' Transformer', '') for m in models], rotation=45, ha='right')
axes[0, 0].set_ylabel('R² Score', fontsize=12, fontweight='bold')
axes[0, 0].set_title('R² Score Comparison', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_df['R²']):
    axes[0, 0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

# RMSE comparison
axes[0, 1].bar(range(len(models)), comparison_df['RMSE (cycles)'], color=colors, alpha=0.7, 
              edgecolor='black', linewidth=1.5)
axes[0, 1].set_xticks(range(len(models)))
axes[0, 1].set_xticklabels([m.replace(' Transformer', '') for m in models], rotation=45, ha='right')
axes[0, 1].set_ylabel('RMSE (cycles)', fontsize=12, fontweight='bold')
axes[0, 1].set_title('RMSE Comparison (Lower is Better)', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_df['RMSE (cycles)']):
    axes[0, 1].text(i, v + max(comparison_df['RMSE (cycles)'])*0.02, f'{v:.2f}', 
                    ha='center', fontweight='bold', fontsize=9)

# MAE comparison
axes[1, 0].bar(range(len(models)), comparison_df['MAE (cycles)'], color=colors, alpha=0.7, 
              edgecolor='black', linewidth=1.5)
axes[1, 0].set_xticks(range(len(models)))
axes[1, 0].set_xticklabels([m.replace(' Transformer', '') for m in models], rotation=45, ha='right')
axes[1, 0].set_ylabel('MAE (cycles)', fontsize=12, fontweight='bold')
axes[1, 0].set_title('MAE Comparison (Lower is Better)', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_df['MAE (cycles)']):
    axes[1, 0].text(i, v + max(comparison_df['MAE (cycles)'])*0.02, f'{v:.2f}', 
                    ha='center', fontweight='bold', fontsize=9)

# Parameter counts
def count_params(model):
    return model.count_params()

param_counts = [
    count_params(model_1_transformer),
    count_params(model_2_transformer),
    count_params(model_3_transformer),
    count_params(model_4_transformer),
    count_params(model_5_transformer)
]

axes[1, 1].bar(range(len(models)), param_counts, color=colors, alpha=0.7, 
              edgecolor='black', linewidth=1.5)
axes[1, 1].set_xticks(range(len(models)))
axes[1, 1].set_xticklabels([m.replace(' Transformer', '') for m in models], rotation=45, ha='right')
axes[1, 1].set_ylabel('Number of Parameters', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Model Complexity (Parameter Count)', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(param_counts):
    axes[1, 1].text(i, v + max(param_counts)*0.01, f'{v:,}', ha='center', 
                   fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

print("✅ Model comparison visualization created!")

In [ ]:
# Plot training history for all models
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Transformer Training Progress: All Model Designs', fontsize=16, fontweight='bold')

histories = [history_1_transformer, history_2_transformer, history_3_transformer, 
             history_4_transformer, history_5_transformer]
labels = ['1-Block', '2-Block', '3-Block', '8-Heads', 'Large Embedding']
colors_plot = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12', '#9B59B6']

# Loss comparison
for hist, label, color in zip(histories, labels, colors_plot):
    axes[0].plot(hist.history['loss'], label=f'{label} (Train)', linestyle='-', linewidth=2, color=color)
    axes[0].plot(hist.history['val_loss'], label=f'{label} (Val)', linestyle='--', linewidth=2, color=color, alpha=0.7)

axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
axes[0].set_title('Loss Comparison', fontsize=13, fontweight='bold')
axes[0].legend(ncol=2, fontsize=9)
axes[0].grid(True, alpha=0.3)

# MAE comparison
for hist, label, color in zip(histories, labels, colors_plot):
    axes[1].plot(hist.history['mae'], label=f'{label} (Train)', linestyle='-', linewidth=2, color=color)
    axes[1].plot(hist.history['val_mae'], label=f'{label} (Val)', linestyle='--', linewidth=2, color=color, alpha=0.7)

axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MAE (cycles)', fontsize=12, fontweight='bold')
axes[1].set_title('MAE Comparison', fontsize=13, fontweight='bold')
axes[1].legend(ncol=2, fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Training progress visualization created!")

## Step 15: Visualize Predictions for All Models

In [ ]:
# Collect all predictions
predictions = {
    '1-Block': y_pred_1_transformer,
    '2-Block': y_pred_2_transformer,
    '3-Block': y_pred_3_transformer,
    '8-Heads': y_pred_4_transformer,
    'Large Embedding': y_pred_5_transformer
}

# Plot predictions vs actual for all models
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('RUL Predictions vs Actual Values - All Transformer Models', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, (name, pred) in enumerate(predictions.items()):
    ax = axes[idx]
    ax.scatter(y_val_split, pred, alpha=0.5, s=20, color=colors[idx])
    min_val = min(y_val_split.min(), pred.min())
    max_val = max(y_val_split.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    ax.set_xlabel('Actual RUL (cycles)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted RUL (cycles)', fontsize=11, fontweight='bold')
    r2_val = comparison_df[comparison_df['Model'] == name + ' Transformer']['R²'].values[0]
    ax.set_title(f'{name}\\n(R² = {r2_val:.4f})', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Remove last subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

print("✅ Predictions visualization created!")

## Step 16: Visualize Attention Mechanism (Conceptual)

Let's visualize how attention works for engine degradation!

In [ ]:
# Visualize attention for engine degradation
# Example: How different cycles attend to each other
cycles = ['Cycle 1', 'Cycle 10', 'Cycle 20', 'Cycle 30']
n_cycles = len(cycles)

# Conceptual attention matrix for engine degradation
# Early cycles might attend more to recent cycles
# Later cycles (near failure) might be more important
attention_matrix_engine = np.array([
    [0.4, 0.3, 0.2, 0.1],  # Cycle 1 attends to (more to itself and recent)
    [0.2, 0.4, 0.3, 0.1],  # Cycle 10 attends to
    [0.1, 0.2, 0.4, 0.3],  # Cycle 20 attends to (more to later cycles)
    [0.05, 0.1, 0.25, 0.6]  # Cycle 30 attends to (strongly to itself - near failure)
])

# Plot attention heatmap
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
im = ax.imshow(attention_matrix_engine, cmap='YlOrRd', aspect='auto')

# Set ticks
ax.set_xticks(np.arange(n_cycles))
ax.set_yticks(np.arange(n_cycles))
ax.set_xticklabels(cycles)
ax.set_yticklabels(cycles)

# Rotate labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add text annotations
for i in range(n_cycles):
    for j in range(n_cycles):
        text = ax.text(j, i, f'{attention_matrix_engine[i, j]:.2f}',
                      ha="center", va="center", color="black", fontweight='bold', fontsize=11)

ax.set_title('Self-Attention for Engine Degradation\\n"How much each cycle attends to other cycles"', 
            fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Attended To (Key)', fontsize=12, fontweight='bold')
ax.set_ylabel('Attending From (Query)', fontsize=12, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Attention Weight', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Attention weights visualization for engine degradation created!")
print("\\n💡 Interpretation:")
print("   - Higher values (darker) = stronger attention")
print("   - Cycle 30 (near failure) strongly attends to itself")
print("   - Early cycles attend more to recent cycles")
print("   - This helps Transformer identify critical degradation patterns!")

## Step 17: Evaluate Best Model on Test Set

Let's test the best Transformer model on unseen engines!

In [ ]:
# Create test sequences
def create_test_sequences(data, sequence_length=30):
    """Create test sequences (last sequence_length timesteps for each engine)"""
    sequences = []
    feature_cols = op_settings + sensors
    
    for unit_id in sorted(data['unit'].unique()):
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        
        # Get last sequence_length timesteps
        if len(unit_data) >= sequence_length:
            sequences.append(unit_features[-sequence_length:])
        else:
            # Pad with first value if sequence is too short
            padding = np.tile(unit_features[0:1], (sequence_length - len(unit_data), 1))
            sequences.append(np.vstack([padding, unit_features]))
    
    return np.array(sequences)

# Create test sequences
X_test_seq = create_test_sequences(test_df, sequence_length)

# Scale test sequences
X_test_reshaped = X_test_seq.reshape(-1, n_features)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test_seq_scaled = X_test_scaled.reshape(X_test_seq.shape[0], n_timesteps, n_features)

# Get actual RUL from the RUL file
y_test_actual = rul_df['RUL'].values

print("✅ Test data prepared!")
print(f"   - Test sequences: {X_test_seq_scaled.shape[0]}")
print(f"   - Actual RUL range: [{y_test_actual.min()}, {y_test_actual.max()}] cycles")

# Use best model based on validation performance
best_models = {
    '1-Block Transformer': model_1_transformer,
    '2-Block Transformer': model_2_transformer,
    '3-Block Transformer': model_3_transformer,
    '8-Heads Transformer': model_4_transformer,
    'Large Embedding Transformer': model_5_transformer
}

best_model = best_models[best_model_name]
y_test_pred = best_model.predict(X_test_seq_scaled, verbose=0)

# Calculate metrics on test set
mse_test = mean_squared_error(y_test_actual, y_test_pred)
mae_test = mean_absolute_error(y_test_actual, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test_actual, y_test_pred)

print(f"\\n📊 Best Model ({best_model_name}) Performance on Test Set:")
print("=" * 60)
print(f"MSE:  {mse_test:.2f}")
print(f"MAE:  {mae_test:.2f} cycles")
print(f"RMSE: {rmse_test:.2f} cycles")
print(f"R²:   {r2_test:.4f}")

print(f"\\n💡 Interpretation:")
print(f"   - On average, predictions are off by {mae_test:.2f} cycles")
print(f"   - The model explains {r2_test*100:.1f}% of the variance in RUL")
print(f"   - This is performance on completely unseen engines!")

In [ ]:
# Plot test set predictions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Test Set RUL Predictions - {best_model_name}', fontsize=16, fontweight='bold')

# Scatter plot: Predictions vs Actual
axes[0].scatter(y_test_actual, y_test_pred, alpha=0.6, s=50, color='blue')
min_val = min(y_test_actual.min(), y_test_pred.min())
max_val = max(y_test_actual.max(), y_test_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual RUL (cycles)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Predicted RUL (cycles)', fontsize=12, fontweight='bold')
axes[0].set_title(f'Predictions vs Actual (R² = {r2_test:.4f})', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error distribution
errors = y_test_actual.flatten() - y_test_pred.flatten()
axes[1].hist(errors, bins=30, alpha=0.7, color='green', edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[1].axvline(x=mae_test, color='blue', linestyle='--', linewidth=2, label=f'MAE = {mae_test:.2f}')
axes[1].axvline(x=-mae_test, color='blue', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error (cycles)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[1].set_title('Error Distribution', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✅ Test set visualization created!")

## 🎓 Summary and Key Takeaways

### ✅ What You've Learned:

1. **Transformer Architecture for RUL**:
   - Self-Attention: Each timestep attends to all timesteps
   - Multi-Head Attention: Multiple attention mechanisms
   - Positional Encoding: Adds temporal position information
   - Transformer Blocks: Stackable attention + feed-forward layers

2. **Transformer Model Designs**:
   - **1-Block**: Simple, fast baseline
   - **2-Block**: Good balance of complexity and performance
   - **3-Block**: Deep, maximum capacity
   - **8-Heads**: More attention heads for diverse patterns
   - **Large Embedding**: More capacity to store information

3. **Key Advantages for RUL**:
   - **Parallel Processing**: Faster than RNNs/LSTMs
   - **Long-range Dependencies**: Captures patterns across many cycles
   - **Attention Visualization**: Can see which cycles are important
   - **Flexibility**: Works well for time series

### 💡 Important Insights:

- **More Blocks ≠ Always Better**: Sometimes simpler is better
- **Attention Heads**: More heads can capture different patterns
- **Embedding Size**: Larger embeddings = more capacity but more parameters
- **Positional Encoding**: Critical for understanding temporal order

### 📚 Transformer vs LSTM/RNN for RUL:

| Aspect | LSTM/RNN | Transformer |
|--------|----------|-------------|
| **Processing** | Sequential | Parallel |
| **Speed** | Slower | Faster |
| **Long-range** | Limited | Excellent |
| **Attention** | Implicit | Explicit |
| **Best For** | Short sequences | Long sequences |

### 🔧 Best Practices:

1. **Start Simple**: Begin with 1-2 Transformer blocks
2. **Attention Heads**: 4-8 heads typically work well
3. **Embedding Dimension**: 64-128 is a good starting point
4. **Positional Encoding**: Always include for time series
5. **Dropout**: Essential for deeper models
6. **Early Stopping**: Monitor validation loss

### 🆚 Model Design Comparison:

| Design | Blocks | Heads | Embedding | Best For |
|--------|--------|-------|-----------|----------|
| **1-Block** | 1 | 4 | 64 | Baseline, simple patterns |
| **2-Block** | 2 | 4 | 64 | Most problems (recommended) |
| **3-Block** | 3 | 4 | 64 | Complex patterns |
| **8-Heads** | 2 | 8 | 64 | Diverse attention patterns |
| **Large Embedding** | 2 | 4 | 128 | Maximum capacity |

### 💡 When to Use Transformers for RUL:

- ✅ Long sequences (many cycles)
- ✅ Complex temporal patterns
- ✅ Need for parallel processing
- ✅ When attention visualization is useful
- ❌ Very short sequences (overkill)
- ❌ Limited computational resources

---

**Great job exploring Transformers for RUL prediction! 🔄✈️✨**